# Milestone 1 — Text Preprocessing & Feature Extraction
**Mata kuliah:** Text & Web Mining
**Dataset:** Tweet berbahasa Indonesia untuk deteksi *hate speech* dan *abusive language*
(Ibrohim & Budi, 2019, *Multi-label Hate Speech and Abusive Language Detection in Indonesian Twitter*, ALW3, hlm. 46–57; lisensi CC BY-NC-SA 4.0).

## Tujuan (objective)
> Mengukur kontribusi tiap tahap preprocessing (perbaikan kata gaul, elongation, negasi, stopword, stemming)
> dan tiap kelompok fitur (n-gram, leksikon kata kasar, fitur konten) terhadap kualitas representasi teks
> untuk mendeteksi hate speech dan abusive language di Twitter Indonesia, dengan evaluasi **per kelas**,
> analisis **waktu proses**, dan pemeriksaan **kebocoran kata kunci**.

## Cara pakai
1. **Google Colab:** upload `data.csv`, `abusive.csv`, `new_kamusalay.csv` ke panel *Files* (ikon folder di kiri), lalu menu *Runtime → Run all*.
2. **Jupyter di laptop:** taruh ketiga file di folder yang sama dengan notebook ini, lalu *Run all*.
3. Seluruh proses memakan waktu sekitar 10–20 menit. Bagian bertanda **(opsional)** bisa dimatikan lewat tombol `RUN_...` di bawah.
4. Jika setelah sel pertama muncul galat seperti *"numpy.dtype size changed"*, pilih *Runtime → Restart session*, lalu *Run all* lagi (sel instalasi tidak perlu diulang).

## Peta notebook (cocok dengan checklist silabus CPMK-1.2)
| Bagian | Isi | Item silabus |
|---|---|---|
| 1–2, 3B | Memuat data & EDA (distribusi label, noise per kelompok, kata khas, mutu leksikon) | – |
| 3 | Pipeline preprocessing | lowercase; hapus tanda baca, angka, HTML, URL, emoji; tokenisasi; stopword; koreksi ejaan; deteksi negasi; koreksi elongation; stemming |
| 4 | POS tagging & lemmatisation **(opsional)** | POS tagging; lemmatisation |
| 5 | Ablation: dampak tiap langkah | – |
| 6 | Feature extraction | Bag of Words; Word2Vec CBoW & Skip-gram; pre-trained; fitur lain (TF-IDF, n-gram, leksikon, konten) |
| 7 | Pemeriksaan kebocoran kata kunci | – |
| 8 | Analisis error | – |
| 9 | Simpan hasil untuk UTS & Milestone 2 | – |

## 0. Persiapan

In [ ]:
%pip install -q PySastrawi gensim scikit-learn pandas matplotlib stanza
import os, re, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from functools import lru_cache
from scipy.sparse import hstack, csr_matrix
from scipy.stats import wilcoxon
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import MaxAbsScaler
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)
os.makedirs("gambar", exist_ok=True)

# --- Tombol-tombol (ubah True/False sesuai kebutuhan) ---
RUN_POS = True            # POS tagging & lemmatisation dengan Stanza (opsional, memakai sampel kecil)
RUN_POS_FEATURES = False  # grup fitur POS untuk SEMUA tweet (opsional, lambat ±10–20 menit; mengikuti paper IEEE Pak Greg)
RUN_PRETRAINED = False    # embedding pre-trained IndoBERT (opsional, unduhan besar ~500 MB)
N_POS_SAMPLE = 300        # jumlah tweet untuk demo POS
N_PRETRAINED = 2000       # jumlah tweet untuk uji IndoBERT

# --- Pustaka opsional: kalau tidak terpasang, bagian terkait dilewati dengan pesan jelas ---
try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    STEMMER = StemmerFactory().create_stemmer()
    HAS_STEM = True
except Exception as e:
    HAS_STEM = False
    print("PySastrawi tidak tersedia -> langkah stemming dilewati:", type(e).__name__)

try:
    from gensim.models import Word2Vec
    HAS_W2V = True
except Exception as e:
    HAS_W2V = False
    print("gensim tidak tersedia -> Word2Vec dilewati:", type(e).__name__)

def find_file(name):
    for d in [".", "/content", "/mnt/user-data/uploads", os.path.expanduser("~/Downloads")]:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"File '{name}' tidak ditemukan. Upload dulu ke folder yang sama dengan notebook.")

## 1. Memuat data
Tiga file dipakai:
- `data.csv` — tweet beserta label (HS = hate speech, Abusive = kata kasar, dan subtipe HS).
- `new_kamusalay.csv` — kamus kata gaul/typo → kata baku (**tanpa header**, jadi dibaca `header=None`).
- `abusive.csv` — daftar 125 kata kasar (dipakai sebagai fitur leksikon).

In [ ]:
df = pd.read_csv(find_file("data.csv"), encoding="latin-1")
kamus = pd.read_csv(find_file("new_kamusalay.csv"), encoding="latin-1", header=None, names=["gaul", "baku"])
abusive_df = pd.read_csv(find_file("abusive.csv"), encoding="utf-8")

print("Jumlah tweet awal   :", len(df))
print("Tweet duplikat      :", df.duplicated("Tweet").sum())
df = df.drop_duplicates(subset="Tweet").reset_index(drop=True)
print("Setelah hapus duplikat:", len(df))
print("Entri kamus gaul    :", len(kamus), "| kata kasar di leksikon:", len(abusive_df))

# Empat kelompok yang dipakai untuk evaluasi per kelas:
#   0 = Netral | 1 = Abusive saja | 2 = HS tanpa kata kasar (hate speech "implisit") | 3 = HS + kata kasar
CLASS_NAMES = ["Netral", "Abusive saja", "HS tanpa kata kasar", "HS + kata kasar"]
df["kelompok"] = df["HS"] * 2 + df["Abusive"]
print()
print(df["kelompok"].map(dict(enumerate(CLASS_NAMES))).value_counts())

## 2. Mengenali "kotoran" di data
Sebelum membersihkan, kita ukur dulu seberapa kotor datanya. Ini jadi alasan kenapa tiap langkah preprocessing perlu ada.

In [ ]:
t = df["Tweet"].astype(str)
kotor = {
    "Mengandung token USER (nama pengguna disamarkan)": t.str.contains(r"\bUSER\b"),
    "Mengandung URL": t.str.contains(r"\bURL\b"),
    "Retweet (RT)": t.str.contains(r"\bRT\b"),
    "Emoji tertulis sebagai kode \\x..": t.str.contains(r"\\x[0-9a-fA-F]{2}"),
    "Kata dipanjangkan (elongation, 3+ huruf sama)": t.str.contains(r"([A-Za-z])\1{2,}"),
    "Mengandung angka": t.str.contains(r"\d"),
    "Mengandung hashtag": t.str.contains("#"),
}
ringkas = pd.Series({k: round(v.mean() * 100, 1) for k, v in kotor.items()}, name="% tweet")
display(ringkas.to_frame())
print("Panjang tweet (jumlah kata):", t.str.split().str.len().describe().round(1).to_dict())

fig, ax = plt.subplots(figsize=(7, 3.2))
df["kelompok"].map(dict(enumerate(CLASS_NAMES))).value_counts().reindex(CLASS_NAMES).plot.barh(ax=ax, color="#4C78A8")
ax.invert_yaxis(); ax.set_xlabel("Jumlah tweet"); ax.set_title("Distribusi kelompok (ada ketidakseimbangan kelas)")
plt.tight_layout(); plt.savefig("gambar/01_distribusi_kelas.png", dpi=150); plt.show()

## 3. Pipeline preprocessing
Setiap langkah dibuat sebagai potongan kecil yang bisa dinyalakan/dimatikan, sehingga nanti bisa diukur pengaruhnya satu per satu (Bagian 5).

| Langkah | Apa yang dilakukan | Kenapa |
|---|---|---|
| Perbaikan artefak | ubah kode `\xf0\x9f..` jadi emoji, buang `\n` dan tanda kutip sisa | data mentah dari Twitter berantakan |
| Hapus RT/USER/URL, HTML, emoji, tanda baca, angka; lowercase | membuat teks seragam | "Bodoh" dan "bodoh" harus dianggap sama |
| **Elongation** | "bangetttt" → "banget" | kata yang dipanjangkan dihitung sebagai kata berbeda |
| **Kata gaul/typo** (koreksi ejaan) | "gak", "jgn", "loe" → bentuk baku via `new_kamusalay.csv` | kata gaul memecah satu makna jadi banyak bentuk |
| **Negasi** | "nggak/gak/ga/tak" → "tidak"; opsional digabung dengan kata sesudahnya ("tidak_bagus") | "tidak bagus" artinya beda dengan "bagus" |
| **Stopword** | buang kata umum ("yang", "di", "dan") | kata umum tidak membawa makna; **kata negasi dan kata ganti orang kedua/ketiga sengaja dipertahankan** karena sering menjadi sasaran hinaan |
| **Stemming** | "memukuli" → "pukul" (Sastrawi) | menyatukan variasi imbuhan |

In [ ]:
EMOJI_RE = re.compile("[\U0001F000-\U0001FAFF☀-➿️‍]")
ELONG_RE = re.compile(r"([a-z])\1{2,}")

SLANG = {str(a).lower(): str(b).lower() for a, b in zip(kamus["gaul"], kamus["baku"])}
NEG_FORMS = {"gak", "ga", "gk", "nggak", "ngga", "nggk", "enggak", "engga", "tak", "tdk", "ndak", "tidak"}
NEG_JOIN = {"tidak", "bukan", "jangan", "belum"}

# Daftar stopword ringkas. Sengaja TIDAK memuat: tidak, bukan, jangan, belum, tanpa, kurang (negasi),
# serta kamu/kau/lu/lo/anda/dia/mereka/kalian (kata ganti yang sering jadi sasaran hinaan).
STOPWORDS = set("""
yang di dan itu dengan untuk dari dalam akan pada juga ke karena tersebut bisa ada lebih kita kami oleh sebagai atau ini
adalah sudah telah dapat sebuah para namun harus setelah hanya saat kepada sehingga secara lain begitu mengapa kenapa yaitu
yakni antara agar bahwa jika jadi apa siapa mana kalau sama masih pun lah kah nya sih dong deh kok lho kan tuh nih sebelum
sedang sedangkan seperti sampai hingga bagi bagaimana dll dsb yg dgn utk dr dlm pd krn tp tapi tetapi serta maka lalu
kemudian ketika sambil selama selalu sering suatu setiap semua seluruh beberapa pernah mau ingin bakal lagi udah dah saya
""".split())

def fix_artifacts(t):
    """Perbaiki sisa-sisa format: kode emoji \\xNN, \\n literal, tanda kutip pembungkus."""
    t = str(t)
    def _dec(m):
        hexes = re.findall(r"\\x([0-9a-fA-F]{2})", m.group(0))
        try:
            return bytes(int(h, 16) for h in hexes).decode("utf-8")
        except Exception:
            return " "
    t = re.sub(r"(?:\\x[0-9a-fA-F]{2})+", _dec, t)
    t = t.replace("\\n", " ").replace("\\t", " ")
    t = re.sub(r"^b['\"]", "", t.strip())
    t = re.sub(r"['\"]$", "", t)
    return t

def basic_clean(t, keep_tags=False):
    t = fix_artifacts(t)
    if not keep_tags:
        t = re.sub(r"\bRT\b", " ", t)
        t = re.sub(r"\bUSER\b", " ", t)
        t = re.sub(r"\bURL\b", " ", t)
    t = re.sub(r"https?://\S+|www\.\S+", " ", t)   # URL asli (jika ada)
    t = re.sub(r"&\w+;", " ", t)                    # kode HTML, mis. &amp;
    t = EMOJI_RE.sub(" ", t)                        # emoji
    t = t.lower()                                   # huruf kecil
    t = re.sub(r"[^a-z\s]", " ", t)                 # tanda baca & angka
    return re.sub(r"\s+", " ", t).strip()

if HAS_STEM:
    @lru_cache(maxsize=None)
    def stem_word(w):
        return STEMMER.stem(w)

def join_neg(toks):
    out, i = [], 0
    while i < len(toks):
        if toks[i] in NEG_JOIN and i + 1 < len(toks) and toks[i + 1] not in NEG_JOIN:
            out.append(toks[i] + "_" + toks[i + 1]); i += 2
        else:
            out.append(toks[i]); i += 1
    return out

def preprocess(t, elong=False, slang=False, neg=None, stop=False, stem=False, keep_tags=False):
    """neg: None (tidak diproses) | 'norm' (seragamkan ke 'tidak') | 'join' (seragamkan + gabung dengan kata sesudahnya)."""
    t = basic_clean(t, keep_tags)
    if elong:
        t = ELONG_RE.sub(r"\1", t)
    toks = t.split()
    if slang:
        out = []
        for w in toks:
            out.extend(SLANG.get(w, w).split())
        toks = out
    if neg:
        toks = ["tidak" if w in NEG_FORMS else w for w in toks]
    if stop:
        toks = [w for w in toks if w not in STOPWORDS]
    if stem and HAS_STEM:
        toks = [stem_word(w) for w in toks]
    if neg == "join":
        toks = join_neg(toks)
    return " ".join(toks)

# --- Demo sebelum/sesudah ---
contoh = df["Tweet"].sample(6, random_state=7).tolist()
contoh.append("USER bagusssss bgt gak nyesel!!! RT URL \\xf0\\x9f\\x98\\xad'")
FULL_DEMO = dict(elong=True, slang=True, neg="join", stop=True, stem=HAS_STEM)
for c in contoh:
    print("ASLI :", c[:140])
    print("BERSIH:", preprocess(c, **FULL_DEMO))
    print()

**Membuang tweet yang kosong.** Beberapa tweet isinya hanya `USER USER USER ...`. Setelah dibersihkan, teksnya kosong, jadi dibuang
supaya semua eksperimen memakai kumpulan tweet yang sama.

In [ ]:
base_txt = [preprocess(x) for x in df["Tweet"]]
keep = np.array([len(s) > 0 for s in base_txt])
print("Tweet kosong setelah pembersihan dasar (dibuang):", int((~keep).sum()))
df = df[keep].reset_index(drop=True)
RAW = df["Tweet"].astype(str).values
y = df["kelompok"].values.astype(int)
print("Jumlah tweet dipakai:", len(df))

## 3B. EDA lanjutan (Exploratory Data Analysis)
EDA = mengenali data sebelum dipakai. Di sini kita menjawab lima pertanyaan, dan semuanya dilihat **per kelompok** (bukan angka total):
1. Seberapa tidak seimbang labelnya, dan subtipe hate speech apa yang paling langka?
2. Apakah label saling berkaitan?
3. Apakah "kotoran" (USER, URL, RT, elongation, negasi) tersebar merata, atau menempel di kelompok tertentu? (petunjuk awal kebocoran)
4. Kata apa yang khas untuk tiap kelompok?
5. Seberapa bagus daftar kata kasar (`abusive.csv`) sebagai penanda?

In [ ]:
# --- (1) Distribusi label & subtipe, (2) keterkaitan antar label ---
SUBTIPE = ["HS_Individual", "HS_Group", "HS_Religion", "HS_Race", "HS_Physical", "HS_Gender", "HS_Other"]
LEVEL = ["HS_Weak", "HS_Moderate", "HS_Strong"]
LABELS = ["HS", "Abusive"] + SUBTIPE + LEVEL

lab_tab = pd.DataFrame({"jumlah": df[LABELS].sum(), "% dari semua tweet": (df[LABELS].mean() * 100).round(1)})
lab_tab["% dari tweet HS"] = [np.nan, np.nan] + [round(df.loc[df.HS == 1, c].mean() * 100, 1) for c in SUBTIPE + LEVEL]
display(lab_tab)

kel_n = pd.Series(y).value_counts()
print(f"Rasio ketidakseimbangan 4 kelompok (terbesar : terkecil) = {kel_n.max() / kel_n.min():.1f} : 1")
print(f"Subtipe terlangka: {lab_tab.loc[SUBTIPE, 'jumlah'].idxmin()} ({int(lab_tab.loc[SUBTIPE, 'jumlah'].min())} tweet); "
      f"terbanyak: {lab_tab.loc[SUBTIPE, 'jumlah'].idxmax()} ({int(lab_tab.loc[SUBTIPE, 'jumlah'].max())} tweet)")
ganjil = int(((df[SUBTIPE + LEVEL].sum(axis=1) > 0) & (df["HS"] == 0)).sum())
print(f"Cek konsistensi: tweet bersubtipe tetapi HS=0 -> {ganjil}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={"width_ratios": [1, 1.25]})
lab_tab.loc[SUBTIPE + LEVEL, "jumlah"].sort_values().plot.barh(ax=axes[0], color="#E15759")
axes[0].set_xlabel("Jumlah tweet"); axes[0].set_title("Subtipe & tingkat hate speech (sangat tidak seimbang)")
corr = df[LABELS].corr()
im = axes[1].imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
axes[1].set_xticks(range(len(LABELS))); axes[1].set_xticklabels(LABELS, rotation=70, ha="right", fontsize=8)
axes[1].set_yticks(range(len(LABELS))); axes[1].set_yticklabels(LABELS, fontsize=8)
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        axes[1].text(j, i, f"{corr.values[i, j]:.1f}", ha="center", va="center", fontsize=6.5,
                     color="white" if abs(corr.values[i, j]) > 0.6 else "black")
axes[1].set_title("Korelasi antar label")
plt.tight_layout(); plt.savefig("gambar/05_label_subtipe_korelasi.png", dpi=150); plt.show()

In [ ]:
# --- (3) Kotoran & panjang tweet PER KELOMPOK (petunjuk awal kebocoran) ---
nama_kel = pd.Series(y).map(dict(enumerate(CLASS_NAMES))).values
low = pd.Series([str(x).lower() for x in RAW])
rawS = pd.Series(RAW)
ciri = pd.DataFrame({
    "Mengandung USER": rawS.str.contains(r"\bUSER\b"),
    "Mengandung URL": rawS.str.contains(r"\bURL\b"),
    "Retweet (RT)": rawS.str.contains(r"\bRT\b"),
    "Emoji berupa kode \\x..": rawS.str.contains(r"\\x[0-9a-fA-F]{2}"),
    "Elongation (3+ huruf sama)": rawS.str.contains(r"([A-Za-z])\1{2,}"),
    "Ada kata negasi": low.str.contains(r"\b(?:tidak|nggak|ngga|gak|ga|bukan|tak|enggak|jangan|belum)\b"),
    "Ada hashtag": rawS.str.contains("#"),
})
ciri["kelompok"] = nama_kel
per_kel = (ciri.groupby("kelompok").mean().T * 100).round(1)[CLASS_NAMES]
per_kel["Selisih terbesar-terkecil"] = (per_kel.max(axis=1) - per_kel.min(axis=1)).round(1)
print("Persentase tweet yang punya ciri tertentu, per kelompok:")
display(per_kel)
print("Ciri dengan selisih antar kelompok paling besar:", per_kel["Selisih terbesar-terkecil"].idxmax(),
      "-> ciri ini bisa jadi 'jalan pintas' bagi model (dicek lagi di Bagian 7).")

panjang = pd.DataFrame({"kata": pd.Series(RAW).str.split().str.len(), "kelompok": nama_kel})
print("\nPanjang tweet (jumlah kata) per kelompok:")
display(panjang.groupby("kelompok")["kata"].describe().loc[CLASS_NAMES].round(1))

fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.boxplot([panjang.loc[panjang.kelompok == c, "kata"] for c in CLASS_NAMES], showfliers=False)
ax.set_xticks(range(1, len(CLASS_NAMES) + 1)); ax.set_xticklabels(CLASS_NAMES, rotation=15)
ax.set_ylabel("Jumlah kata per tweet"); ax.set_title("Panjang tweet per kelompok")
plt.tight_layout(); plt.savefig("gambar/06_panjang_per_kelompok.png", dpi=150); plt.show()

In [ ]:
# --- (4) Kata yang paling sering & paling KHAS untuk tiap kelompok ---
from collections import Counter
EDA_TXT = [preprocess(x, elong=True, slang=True, neg="norm", stop=True) for x in RAW]
df_cnt = {k: Counter() for k in range(4)}
for s, k in zip(EDA_TXT, y):
    df_cnt[k].update(set(s.split()))                  # dihitung per tweet (document frequency)
n_k = {k: int((y == k).sum()) for k in range(4)}
all_cnt = sum(df_cnt.values(), Counter())

def kata_khas(k, top=10, min_total=15):
    """Skor = log-odds: seberapa jauh kata lebih sering muncul di kelompok k dibanding kelompok lain."""
    n_rest = len(y) - n_k[k]
    rows = []
    for w, a in df_cnt[k].items():
        b = all_cnt[w] - a
        if a >= 5 and a + b >= min_total:
            rows.append((w, np.log((a + .5) / (n_k[k] - a + .5)) - np.log((b + .5) / (n_rest - b + .5)), a))
    return sorted(rows, key=lambda r: -r[1])[:top]

print("Kata PALING KHAS tiap kelompok (log-odds; kata yang sering di semua kelompok tidak akan muncul di sini):\n")
for k, nama in enumerate(CLASS_NAMES):
    print(f"- {nama}: " + ", ".join(f"{w}" for w, _, _ in kata_khas(k)))

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for k, ax in enumerate(axes.ravel()):
    top = df_cnt[k].most_common(10)
    ax.barh([w for w, _ in top][::-1], [c / n_k[k] * 100 for _, c in top][::-1], color="#4C78A8")
    ax.set_title(f"{CLASS_NAMES[k]} (n={n_k[k]})", fontsize=10); ax.set_xlabel("% tweet yang memuat kata")
plt.suptitle("10 kata tersering per kelompok: banyak yang sama antar kelompok, itu sebabnya perlu ukuran 'khas'", fontsize=10)
plt.tight_layout(); plt.savefig("gambar/07_kata_per_kelompok.png", dpi=150); plt.show()

In [ ]:
# --- (5) Seberapa bagus daftar kata kasar (abusive.csv)? ---
AB_SET = {str(w).strip().lower() for w in abusive_df["ABUSIVE"].dropna()}
AB_ONE = {w for w in AB_SET if " " not in w}
EDA_LEX = [preprocess(x, elong=True, slang=True, neg="norm") for x in RAW]
ab_per_tweet = [set(s.split()) & AB_ONE for s in EDA_LEX]
muncul = Counter(w for ws in ab_per_tweet for w in ws)

print(f"Dari {len(AB_ONE)} kata tunggal di leksikon, {len(AB_ONE) - len(muncul)} tidak pernah muncul di data ini.")
print("\nKata leksikon yang paling sering muncul:", ", ".join(f"{w} ({c})" for w, c in muncul.most_common(12)))

ada = np.array([len(ws) > 0 for ws in ab_per_tweet])
tabel = pd.DataFrame({"% tweet berisi kata leksikon": [round(ada[y == k].mean() * 100, 1) for k in range(4)]}, index=CLASS_NAMES)
display(tabel)
presisi = {}
for w in muncul:
    ada_w = np.array([w in ws for ws in ab_per_tweet])
    presisi[w] = (ada_w.sum(), (y[ada_w] % 2 == 1).mean())   # y%2==1 -> label Abusive (kelompok 1 atau 3)
tinggi = sorted([(w, n, p) for w, (n, p) in presisi.items() if n >= 40], key=lambda r: r[2])
print("\nKata leksikon yang LEMAH sebagai penanda Abusive (sering muncul, tapi jarang berlabel Abusive):")
for w, n, p in tinggi[:8]:
    print(f"  {w:12s} muncul {n:4d}x, hanya {p:.0%} berlabel Abusive")
print("\nKata leksikon yang KUAT sebagai penanda Abusive:")
for w, n, p in tinggi[::-1][:8]:
    print(f"  {w:12s} muncul {n:4d}x, {p:.0%} berlabel Abusive")

**Catatan untuk slide (isi dengan hasil EDA-mu sendiri):**
- Subtipe hate speech yang paling langka → bahan alasan perlu evaluasi per kelas dan penanganan imbalance di Milestone 2.
- Ciri "kotoran" yang tidak merata antar kelompok → bahan tes kebocoran di Bagian 7.
- Kata khas tiap kelompok → bahan awal untuk aspek di UTS dan gambaran apa yang ditangkap fitur.
- Kata leksikon yang lemah sebagai penanda → alasan kenapa leksikon saja tidak cukup.

## 4. POS tagging & lemmatisation (opsional)
POS tagging memberi label jenis kata (kata benda, kata sifat, dst.). Berguna untuk **UTS**: kata benda adalah kandidat *aspek*.
Untuk bahasa Indonesia dipakai **Stanza** pada sampel kecil (model akan diunduh otomatis, butuh internet).
Catatan: untuk bahasa Indonesia, stemming (Sastrawi) dan lemmatisation hasilnya mirip; stemming dipakai di pipeline utama karena jauh lebih cepat.

In [ ]:
LIGHT = [preprocess(x, elong=True, slang=True, neg="norm") for x in RAW]   # versi "ringan": tanpa stopword/stemming

if RUN_POS:
    try:
        import stanza
        stanza.download("id", verbose=False)
        nlp_pos = stanza.Pipeline("id", processors="tokenize,pos,lemma", tokenize_pretokenized=True, verbose=False)
        idx = np.random.RandomState(SEED).choice(len(LIGHT), size=min(N_POS_SAMPLE, len(LIGHT)), replace=False)
        sents = [LIGHT[i].split() for i in idx if LIGHT[i].strip()]
        t0 = time.time()
        doc = nlp_pos(sents)
        print(f"POS tagging {len(sents)} tweet selesai dalam {time.time()-t0:.1f} detik\n")
        rows = [(w.text, w.upos, w.lemma) for s in doc.sentences for w in s.words]
        pos_df = pd.DataFrame(rows, columns=["kata", "POS", "lemma"])
        for s in doc.sentences[:3]:
            print([(w.text, w.upos, w.lemma) for w in s.words])
        print("\nDistribusi POS:"); display(pos_df["POS"].value_counts().head(8).to_frame())
        print("Kata benda (NOUN) paling sering -> kandidat aspek untuk UTS:")
        display(pos_df[pos_df["POS"] == "NOUN"]["lemma"].value_counts().head(15).to_frame("frekuensi"))
    except Exception as e:
        print("POS tagging dilewati:", type(e).__name__, str(e)[:150])
else:
    print("RUN_POS = False -> bagian POS dilewati.")

## 5. Ablation: seberapa besar pengaruh tiap langkah preprocessing?
**Cara kerjanya:** kita ukur kualitas representasi teks dengan sebuah "penggaris", yaitu model sederhana (*Linear SVM* di atas fitur TF-IDF).
Penggaris yang sama dipakai untuk semua konfigurasi, jadi perbedaan skor berasal dari preprocessing-nya.

- **Skor** = F1-macro: rata-rata F1 dari keempat kelompok, sehingga kelompok kecil tidak "tenggelam" (akurasi biasa bisa menipu pada data tidak seimbang).
- **Validasi:** 10-fold cross-validation (10 skor), sama seperti protokol di paper-paper Pak Greg. Vektorisasi dipelajari hanya dari data latih tiap lipatan, supaya tidak ada kebocoran ke data uji.
- **p-value (Wilcoxon):** p < 0,05 artinya perbedaan terhadap konfigurasi dasar kemungkinan besar bukan kebetulan.
- **Kosakata:** jumlah kata unik. Semakin kecil, semakin ringkas representasinya (salah satu manfaat preprocessing selain skor).
- **Waktu:** biaya proses tiap langkah per 300 tweet, supaya kelihatan apakah langkah mahal sebanding manfaatnya.

In [ ]:
def cv_run(texts=None, y=None, dense=None, ngram=(1, 1), weighting="tfidf", max_features=5000,
           model="svm", n_splits=10, n_repeats=1):
    rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=SEED)
    fold_scores = []
    oof = np.zeros((n_repeats, len(y)), dtype=int)
    for k, (tr, te) in enumerate(rskf.split(np.zeros(len(y)), y)):
        tr_parts, te_parts = [], []
        if texts is not None:
            if weighting == "count":      # Bag of Words: hitungan mentah
                vec = CountVectorizer(ngram_range=ngram, max_features=max_features, token_pattern=r"\S+", lowercase=False)
            else:                          # "tf" = frekuensi ternormalisasi tanpa IDF; "tfidf" = dengan IDF
                vec = TfidfVectorizer(ngram_range=ngram, max_features=max_features, token_pattern=r"\S+",
                                      lowercase=False, use_idf=(weighting == "tfidf"))
            tr_parts.append(vec.fit_transform(texts[tr])); te_parts.append(vec.transform(texts[te]))
        if dense is not None:
            sc = MaxAbsScaler().fit(dense[tr])
            tr_parts.append(csr_matrix(sc.transform(dense[tr]))); te_parts.append(csr_matrix(sc.transform(dense[te])))
        Xtr, Xte = hstack(tr_parts).tocsr(), hstack(te_parts).tocsr()
        clf = LinearSVC(C=1.0, max_iter=5000, random_state=SEED) if model == "svm" else MultinomialNB()
        clf.fit(Xtr, y[tr]); p = clf.predict(Xte)
        fold_scores.append(f1_score(y[te], p, average="macro"))
        oof[k // n_splits, te] = p
    per_class = np.mean([f1_score(y, oof[r], average=None, labels=[0, 1, 2, 3]) for r in range(n_repeats)], axis=0)
    return dict(macro=float(np.mean(fold_scores)), std=float(np.std(fold_scores)),
                per_class=per_class, folds=fold_scores, oof=oof[0])

def make_texts(cfg, **kw):
    return np.array([preprocess(x, **cfg, **kw) for x in RAW])

def time_cfg(cfg, n=300):
    """Ukur waktu proses n tweet. Stemming diukur TANPA cache supaya biayanya jujur."""
    global stem_word
    saved = stem_word if HAS_STEM else None
    if HAS_STEM:
        stem_word = lambda w: STEMMER.stem(w)
    sub = RAW[:n]
    t0 = time.time()
    for x in sub:
        preprocess(x, **cfg)
    dur = time.time() - t0
    if HAS_STEM:
        stem_word = saved
    return dur

ALL = dict(elong=True, slang=True, neg="join", stop=True, stem=HAS_STEM)
configs = {"0. Dasar (tanpa langkah tambahan)": dict()}
configs["+ elongation"] = dict(elong=True)
configs["+ kata gaul/typo"] = dict(slang=True)
configs["+ negasi (seragamkan)"] = dict(neg="norm")
configs["+ negasi (gabung kata)"] = dict(neg="join")
configs["+ stopword"] = dict(stop=True)
if HAS_STEM:
    configs["+ stemming"] = dict(stem=True)
configs["SEMUA langkah"] = ALL
off = {"elong": False, "slang": False, "neg": None, "stop": False, "stem": False}
for key, label in [("elong", "elongation"), ("slang", "kata gaul"), ("neg", "negasi"), ("stop", "stopword"), ("stem", "stemming")]:
    if key == "stem" and not HAS_STEM:
        continue
    configs[f"SEMUA tanpa {label}"] = {**ALL, key: off[key]}

results, texts_by_cfg = {}, {}
for name, cfg in configs.items():
    txt = make_texts(cfg)
    texts_by_cfg[name] = txt
    r = cv_run(txt, y)
    r["detik"] = time_cfg(cfg)
    r["vocab"] = len(set(" ".join(txt).split()))
    results[name] = r
    print(f"{name:38s} F1-macro = {r['macro']:.4f} ± {r['std']:.4f}   kosakata = {r['vocab']:6d}   waktu = {r['detik']:.2f} dtk/300 tweet")

base_name = "0. Dasar (tanpa langkah tambahan)"
rows = []
for name, r in results.items():
    try:
        p = wilcoxon(r["folds"], results[base_name]["folds"]).pvalue if name != base_name else np.nan
    except ValueError:
        p = np.nan
    rows.append({"Konfigurasi": name, "F1-macro": r["macro"], "±": r["std"],
                 "Δ vs dasar": r["macro"] - results[base_name]["macro"], "p-value": p,
                 "Kosakata": r["vocab"], "Waktu (dtk/300 tweet)": r["detik"]})
abl = pd.DataFrame(rows).set_index("Konfigurasi")
display(abl.round(4))

fig, ax = plt.subplots(figsize=(8, 0.42 * len(abl) + 1.2))
ax.barh(abl.index, abl["F1-macro"], color=["#999999" if n == base_name else "#4C78A8" for n in abl.index], xerr=abl["±"], capsize=2)
ax.invert_yaxis()
lo = max(0.0, abl["F1-macro"].min() - 0.05)
ax.set_xlim(lo, abl["F1-macro"].max() + 0.04); ax.set_xlabel("F1-macro (lebih tinggi lebih baik)")
ax.axvline(results[base_name]["macro"], color="#999999", ls="--", lw=1)
ax.set_title("Pengaruh tiap langkah preprocessing")
plt.tight_layout(); plt.savefig("gambar/02_ablation_preprocessing.png", dpi=150); plt.show()

BEST_NAME = abl["F1-macro"].idxmax()
BEST_CFG = configs[BEST_NAME]
BEST_TEXTS = texts_by_cfg[BEST_NAME]
print("Konfigurasi terbaik:", BEST_NAME, "->", BEST_CFG)

**Cara membaca tabel di atas:** kolom *Δ vs dasar* positif berarti langkah itu membantu; negatif berarti merugikan.
Selisih yang lebih kecil dari kolom `±` (simpangan baku antar lipatan) praktis dianggap seri, dan p-value besar menegaskan hal itu.
Jangan kaget kalau ada langkah yang tidak membantu atau malah merugikan. Itu justru temuan yang bagus dan harus dijelaskan alasannya
(misalnya kata gaul yang "dibakukan", stopword, dan stemming bisa ikut menghapus sinyal yang penting untuk hinaan).
Bandingkan juga kolom *Kosakata*: langkah yang tidak menaikkan skor tetap bisa berguna karena membuat representasi lebih ringkas.

## 6. Feature extraction: mengubah teks jadi angka
Kelompok fitur yang dibandingkan (dengan preprocessing terbaik dari Bagian 5):
- **A. Frekuensi kata:** Bag of Words (hitungan kata), TF-IDF, dan n-gram (1 kata, 1–2 kata, 1–3 kata), dibatasi 5.000 fitur teratas (angka yang juga dipakai di paper-paper Pak Greg).
- **POS (opsional):** proporsi tiap jenis kata per tweet; di paper IEEE Pak Greg, grup ini memberi peningkatan terbesar.
- **B. Leksikon:** jumlah/rasio kata kasar dari `abusive.csv`.
- **C. Konten:** panjang tweet, jumlah huruf kapital, tanda seru, emoji, elongation, dan sebagainya.
- **D. Embedding:** Word2Vec (CBoW dan Skip-gram) dilatih dari tweet sendiri, ditambah model pre-trained (opsional).

In [ ]:
ABUSIVE = [str(w).strip().lower() for w in abusive_df["ABUSIVE"].dropna()]
ABUSIVE_SINGLE = {w for w in ABUSIVE if " " not in w}
ABUSIVE_PHRASE = [w for w in ABUSIVE if " " in w]

def lexicon_features(light_text):
    toks = light_text.split()
    n = sum(w in ABUSIVE_SINGLE for w in toks)
    padded = " " + light_text + " "
    n += sum((" " + p + " ") in padded for p in ABUSIVE_PHRASE)
    return {"n_kasar": n, "rasio_kasar": n / max(len(toks), 1), "ada_kasar": int(n > 0)}

def content_features(raw):
    r = fix_artifacts(raw)
    letters = re.sub(r"\b(USER|URL|RT)\b", "", r)
    alpha = [c for c in letters if c.isalpha()]
    return {
        "n_kata": len(r.split()), "n_char": len(r),
        "n_user": len(re.findall(r"\bUSER\b", r)), "n_url": len(re.findall(r"\bURL\b", r)),
        "ada_rt": int(bool(re.search(r"\bRT\b", r))), "n_hashtag": r.count("#"),
        "n_emoji": len(EMOJI_RE.findall(r)), "rasio_kapital": sum(c.isupper() for c in alpha) / max(len(alpha), 1),
        "n_seru": r.count("!"), "n_tanya": r.count("?"),
        "n_elongasi": len(re.findall(r"([A-Za-z])\1{2,}", r)), "n_angka": len(re.findall(r"\d", r)),
    }

lex_df = pd.DataFrame([lexicon_features(s) for s in LIGHT])
con_df = pd.DataFrame([content_features(x) for x in RAW])
con_df["n_negasi"] = [sum(w in NEG_JOIN for w in s.split()) for s in LIGHT]
LEX, CON = lex_df.values.astype(float), con_df.values.astype(float)

print("Fitur leksikon:", list(lex_df.columns))
print("Fitur konten  :", list(con_df.columns))
print("\nRata-rata fitur leksikon per kelompok:")
display(lex_df.groupby(y).mean().rename(index=dict(enumerate(CLASS_NAMES))).round(3))

In [ ]:
# --- Word2Vec (CBoW & Skip-gram) dilatih dari tweet sendiri ---
def mean_vectors(model, texts):
    dim = model.vector_size
    out = np.zeros((len(texts), dim), dtype=float)
    for i, s in enumerate(texts):
        v = [model.wv[w] for w in s.split() if w in model.wv]
        if v:
            out[i] = np.mean(v, axis=0)
    return out

EMB = {}
W2V_MODELS = {}
if HAS_W2V:
    sentences = [s.split() for s in BEST_TEXTS]
    for nama, sg in [("Word2Vec CBoW", 0), ("Word2Vec Skip-gram", 1)]:
        m = Word2Vec(sentences, vector_size=100, window=5, min_count=2, sg=sg, workers=1, epochs=15, seed=SEED)
        W2V_MODELS[nama] = m
        EMB[nama] = mean_vectors(m, BEST_TEXTS)
        print(nama, "-> ukuran kosakata:", len(m.wv))
    m = W2V_MODELS["Word2Vec Skip-gram"]
    for kata in ["bodoh", "agama", "cina"]:
        if kata in m.wv:
            print(f"Kata paling mirip dengan '{kata}':", [w for w, _ in m.wv.most_similar(kata, topn=6)])
else:
    print("gensim tidak tersedia -> Word2Vec dilewati (di Colab otomatis aktif).")

In [ ]:
# --- (opsional) Embedding pre-trained IndoBERT, memakai subset tweet ---
PRE_IDX, PRE_EMB = None, None
if RUN_PRETRAINED:
    try:
        import torch
        from transformers import AutoTokenizer, AutoModel
        tok = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")
        bert = AutoModel.from_pretrained("indobenchmark/indobert-base-p1").eval()
        PRE_IDX = np.random.RandomState(SEED).choice(len(LIGHT), size=min(N_PRETRAINED, len(LIGHT)), replace=False)
        vecs = []
        with torch.no_grad():
            for i in range(0, len(PRE_IDX), 32):
                batch = [LIGHT[j] for j in PRE_IDX[i:i + 32]]
                enc = tok(batch, padding=True, truncation=True, max_length=64, return_tensors="pt")
                out = bert(**enc).last_hidden_state
                mask = enc["attention_mask"].unsqueeze(-1)
                vecs.append(((out * mask).sum(1) / mask.sum(1)).numpy())
        PRE_EMB = np.vstack(vecs)
        print("Embedding IndoBERT selesai:", PRE_EMB.shape)
    except Exception as e:
        print("IndoBERT dilewati:", type(e).__name__, str(e)[:150])
else:
    print("RUN_PRETRAINED = False -> embedding pre-trained dilewati.")

In [ ]:
# --- (opsional) Grup fitur POS: proporsi tiap jenis kata per tweet, untuk SEMUA tweet ---
# Di paper IEEE Pak Greg, grup fitur POS memberi peningkatan terbesar. Lambat, jadi default mati (RUN_POS_FEATURES = False).
from collections import Counter
POSF = None
if RUN_POS_FEATURES:
    try:
        import stanza
        stanza.download("id", verbose=False)
        nlp_all = stanza.Pipeline("id", processors="tokenize,pos", tokenize_pretokenized=True, verbose=False)
        UPOS = ["ADJ", "ADP", "ADV", "AUX", "CCONJ", "DET", "INTJ", "NOUN", "NUM", "PART", "PRON", "PROPN", "SCONJ", "SYM", "VERB", "X", "PUNCT"]
        rows_pos, t0 = [], time.time()
        for i in range(0, len(LIGHT), 500):
            chunk = [(s.split() or ["kosong"]) for s in LIGHT[i:i + 500]]
            for sent in nlp_all(chunk).sentences:
                tags = [w.upos for w in sent.words]
                c = Counter(tags)
                n = max(len(tags), 1)
                rows_pos.append([c.get(u, 0) / n for u in UPOS])
            print(f"POS {min(i + 500, len(LIGHT))}/{len(LIGHT)}  ({time.time() - t0:.0f} dtk)")
        POSF = np.array(rows_pos)
        assert len(POSF) == len(LIGHT)
        print("Fitur POS siap:", POSF.shape)
    except Exception as e:
        POSF = None
        print("Fitur POS dilewati:", type(e).__name__, str(e)[:150])
else:
    print("RUN_POS_FEATURES = False -> grup fitur POS dilewati.")

In [ ]:
# --- Perbandingan semua kelompok fitur ---
feat_runs = [
    ("A1. Bag of Words (hitungan kata) + Naive Bayes", dict(texts=BEST_TEXTS, weighting="count", model="nb")),
    ("A1b. Bag of Words (hitungan kata) + SVM", dict(texts=BEST_TEXTS, weighting="count")),
    ("A1c. TF ternormalisasi (tanpa IDF) + SVM", dict(texts=BEST_TEXTS, weighting="tf")),
    ("A2. TF-IDF (1 kata)", dict(texts=BEST_TEXTS)),
    ("A3. TF-IDF n-gram 1–2", dict(texts=BEST_TEXTS, ngram=(1, 2))),
    ("A4. TF-IDF n-gram 1–3", dict(texts=BEST_TEXTS, ngram=(1, 3))),
    ("B.  Leksikon kata kasar saja", dict(dense=LEX)),
    ("C.  Fitur konten saja", dict(dense=CON)),
    ("A2+B. TF-IDF + leksikon", dict(texts=BEST_TEXTS, dense=LEX)),
    ("A2+C. TF-IDF + konten", dict(texts=BEST_TEXTS, dense=CON)),
    ("A2+B+C. TF-IDF + leksikon + konten", dict(texts=BEST_TEXTS, dense=np.hstack([LEX, CON]))),
]
if POSF is not None:
    feat_runs.append(("A2+B+C+POS. TF-IDF + leksikon + konten + POS", dict(texts=BEST_TEXTS, dense=np.hstack([LEX, CON, POSF]))))
for nama, E in EMB.items():
    feat_runs.append((f"D.  {nama}", dict(dense=E)))

feat_rows, feat_res = [], {}
for nama, kw in feat_runs:
    r = cv_run(y=y, **kw)
    feat_res[nama] = r
    feat_rows.append({"Fitur": nama, "F1-macro": r["macro"], "±": r["std"],
                      **{f"F1 {c}": v for c, v in zip(CLASS_NAMES, r["per_class"])}})
feat_tab = pd.DataFrame(feat_rows).set_index("Fitur")
display(feat_tab.round(3))

if PRE_EMB is not None:
    # IndoBERT diuji pada subset; bandingkan adil dengan TF-IDF pada subset yang sama
    ysub = y[PRE_IDX]
    r_pre = cv_run(y=ysub, dense=PRE_EMB)
    r_tf = cv_run(texts=BEST_TEXTS[PRE_IDX], y=ysub)
    print(f"Subset {len(PRE_IDX)} tweet -> IndoBERT: {r_pre['macro']:.3f} | TF-IDF: {r_tf['macro']:.3f}")

fig, ax = plt.subplots(figsize=(8, 0.42 * len(feat_tab) + 1.2))
ax.barh(feat_tab.index, feat_tab["F1-macro"], color="#59A14F", xerr=feat_tab["±"], capsize=2)
ax.invert_yaxis(); ax.set_xlabel("F1-macro"); ax.set_title("Perbandingan kelompok fitur")
plt.tight_layout(); plt.savefig("gambar/03_perbandingan_fitur.png", dpi=150); plt.show()

BEST_FEAT = feat_tab["F1-macro"].idxmax()
print("Fitur terbaik:", BEST_FEAT)

# --- Menguji pernyataan di paper Pak Greg (Resampling, Bagian 4.2): TF-IDF dikatakan lebih condong ke kelas mayoritas daripada TF ---
# Pembanding adil: classifier sama (SVM), hanya pembobotan yang beda. "TF ternormalisasi" dipakai supaya selisihnya murni karena IDF,
# bukan karena normalisasi panjang. Lihat apakah kelompok kecil dirugikan oleh TF-IDF.
kol = [f"F1 {c}" for c in CLASS_NAMES]
tf_vs = feat_tab.loc[["A1b. Bag of Words (hitungan kata) + SVM", "A1c. TF ternormalisasi (tanpa IDF) + SVM",
                      "A2. TF-IDF (1 kata)"], kol].T
tf_vs.columns = ["BoW hitungan", "TF ternormalisasi", "TF-IDF"]
tf_vs["Selisih (TF-IDF - TF ternormalisasi)"] = tf_vs["TF-IDF"] - tf_vs["TF ternormalisasi"]
tf_vs["Jumlah tweet"] = [int((y == k).sum()) for k in range(4)]
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
print("\nPembobotan per kelompok (F1):")
display(tf_vs.round(3))
rugi = tf_vs.loc[tf_vs["Selisih (TF-IDF - TF ternormalisasi)"] < 0].index.tolist()
print("Kelompok yang F1-nya TURUN saat memakai IDF:", [r.replace("F1 ", "") for r in rugi] if rugi else "tidak ada")

**Hasil per kelompok** ada di kolom `F1 ...` pada tabel di atas. Cari kelompok dengan F1 terendah, lalu cocokkan dengan *confusion matrix*
di Bagian 8 untuk melihat ia sering tertukar dengan kelompok mana. Perhatikan juga kelompok **HS tanpa kata kasar**: tidak ada kata kasar
di dalamnya, jadi fitur leksikon (B) tidak membantu, dan sebagian tweetnya terbaca "Netral".

## 7. Pemeriksaan kebocoran kata kunci
Model kadang terlihat pintar karena "menyontek" petunjuk yang kebetulan ada di data. Tiga pemeriksaan:
1. Apa pengaruh token `USER/RT/URL` jika tidak dibuang?
2. Seberapa besar model bergantung pada kata dari daftar kata kasar? (kata-kata itu dibuang, lalu skornya dibandingkan)
3. Kata apa yang paling menentukan tiap kelompok?

In [ ]:
cfg_nostem = {**BEST_CFG, "stem": False}
txt_clean = make_texts(cfg_nostem)
txt_tags = make_texts(cfg_nostem, keep_tags=True)
txt_nolex = np.array([" ".join(w for w in s.split() if w not in ABUSIVE_SINGLE) for s in txt_clean])

r_clean, r_tags, r_nolex = cv_run(txt_clean, y), cv_run(txt_tags, y), cv_run(txt_nolex, y)
leak = pd.DataFrame({
    "Skenario": ["Normal (USER/RT/URL dibuang)", "USER/RT/URL DIPERTAHANKAN", "Kata dari daftar kasar DIBUANG"],
    "F1-macro": [r_clean["macro"], r_tags["macro"], r_nolex["macro"]],
}).set_index("Skenario")
leak["Δ vs normal"] = leak["F1-macro"] - r_clean["macro"]
display(leak.round(4))
print("Kelompok yang paling terdampak saat kata kasar dibuang (F1 per kelompok):")
display(pd.DataFrame({"Normal": r_clean["per_class"], "Tanpa kata kasar": r_nolex["per_class"]}, index=CLASS_NAMES).round(3))

vec = TfidfVectorizer(max_features=5000, token_pattern=r"\S+", lowercase=False)
Xall = vec.fit_transform(txt_clean)
svm_all = LinearSVC(C=1.0, max_iter=5000, random_state=SEED).fit(Xall, y)
vocab = np.array(vec.get_feature_names_out())
print("\nKata paling berpengaruh per kelompok:")
for k, nama in enumerate(CLASS_NAMES):
    top = vocab[np.argsort(svm_all.coef_[k])[::-1][:12]]
    print(f"- {nama}: {', '.join(top)}")

## 8. Analisis error
Di mana model salah? Kita lihat *confusion matrix* (baris = label sebenarnya, kolom = tebakan) dan contoh kesalahan terbesar.

In [ ]:
best_kw = dict(feat_runs)[BEST_FEAT] if BEST_FEAT in dict(feat_runs) else dict(texts=BEST_TEXTS)
oof = feat_res[BEST_FEAT]["oof"]
cm = pd.crosstab(pd.Series(y, name="Sebenarnya").map(dict(enumerate(CLASS_NAMES))),
                 pd.Series(oof, name="Tebakan").map(dict(enumerate(CLASS_NAMES))))
cm = cm.reindex(index=CLASS_NAMES, columns=CLASS_NAMES, fill_value=0)
display(cm)

cmn = cm.values / cm.values.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(6, 4.6))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(4)); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right")
ax.set_yticks(range(4)); ax.set_yticklabels(CLASS_NAMES)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", color="white" if cmn[i, j] > 0.5 else "black")
ax.set_xlabel("Tebakan"); ax.set_ylabel("Sebenarnya"); ax.set_title(f"Confusion matrix ({BEST_FEAT.split('.')[0]})")
plt.tight_layout(); plt.savefig("gambar/04_confusion_matrix.png", dpi=150); plt.show()

print("\nContoh: tweet 'HS tanpa kata kasar' yang tidak berhasil dikenali (catatan: isi dataset memuat bahasa kasar):")
salah = np.where((y == 2) & (oof != 2))[0]
for i in salah[:8]:
    print(f"- [{CLASS_NAMES[oof[i]]}] {fix_artifacts(RAW[i])[:150]}")

## 9. Simpan hasil untuk UTS & Milestone 2
File-file ini bisa dipakai lagi tanpa mengulang preprocessing.

In [ ]:
out = pd.DataFrame({
    "tweet_asli": RAW,
    "teks_bersih": BEST_TEXTS,           # konfigurasi preprocessing terbaik
    "teks_ringan": LIGHT,                # tanpa stopword/stemming (cocok untuk POS & analisis aspek di UTS)
    "kelompok": [CLASS_NAMES[i] for i in y],
})
out = pd.concat([out, df[[c for c in df.columns if c not in ("Tweet", "kelompok")]].reset_index(drop=True),
                 lex_df.reset_index(drop=True), con_df.reset_index(drop=True)], axis=1)
out.to_csv("data_bersih.csv", index=False, encoding="utf-8")
abl.round(4).to_csv("hasil_ablation.csv")
feat_tab.round(4).to_csv("hasil_fitur.csv")
print("Tersimpan: data_bersih.csv, hasil_ablation.csv, hasil_fitur.csv, dan gambar/*.png")

## 10. Ringkasan otomatis (bahan slide)

In [ ]:
base = results[base_name]["macro"]
print(f"Jumlah tweet dipakai        : {len(df)}")
print(f"Konfigurasi preprocessing terbaik : {BEST_NAME}")
print(f"F1-macro dasar -> terbaik   : {base:.3f} -> {results[BEST_NAME]['macro']:.3f}  (Δ = {results[BEST_NAME]['macro']-base:+.3f})")
bantu = abl.drop(index=[base_name, "SEMUA langkah"], errors="ignore")
bantu = bantu[bantu.index.str.startswith("+")]
if len(bantu):
    print("Langkah tambahan paling membantu :", bantu["Δ vs dasar"].idxmax(), f"({bantu['Δ vs dasar'].max():+.3f})")
    print("Langkah tambahan paling merugikan:", bantu["Δ vs dasar"].idxmin(), f"({bantu['Δ vs dasar'].min():+.3f})")
print(f"Fitur terbaik               : {BEST_FEAT} (F1-macro {feat_tab.loc[BEST_FEAT, 'F1-macro']:.3f})")
pc = feat_tab.loc[BEST_FEAT, [f"F1 {c}" for c in CLASS_NAMES]].astype(float)
print("F1 per kelompok (fitur terbaik):", {c: round(float(pc[f'F1 {c}']), 3) for c in CLASS_NAMES})
print("Kelompok tersulit (F1 terendah):", pc.idxmin().replace("F1 ", ""))
impl = cm.loc["HS tanpa kata kasar"]
print(f"HS tanpa kata kasar yang terbaca 'Netral': {impl['Netral'] / impl.sum():.1%}")
print(f"Pengaruh mempertahankan USER/RT/URL     : {leak.loc['USER/RT/URL DIPERTAHANKAN', 'Δ vs normal']:+.3f} F1-macro")
print(f"Dampak membuang kata kasar  : {leak.loc['Kata dari daftar kasar DIBUANG', 'Δ vs normal']:+.3f} F1-macro")